In [1]:
"""
Used for seq2seq and mice based on dataset flights
"""
import os
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import math
import copy
import pandas as pd
from torch.utils.data import DataLoader, Dataset, random_split
import GPUtil
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from torch.optim import AdamW, SGD, Adam
import numpy as np
from sklearn.model_selection import train_test_split

# preprocess
dirty_path = '../dataset/flight/dirty.csv'
clean_path = '../dataset/flight/clean.csv'

dirty_df = pd.read_csv(dirty_path)
clean_df = pd.read_csv(clean_path)
clean_df.columns = dirty_df.columns

FileNotFoundError: [Errno 2] No such file or directory: '../dataset/flight/dirty.csv'

In [2]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name())
print(torch.cuda.current_device())
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

2.6.0
True
8
NVIDIA RTX A5000
0
cuda:0


In [3]:
class Seq2SeqDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.data = data
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_text, target_text = self.data[idx]
        inputs = self.tokenizer.encode_plus(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        targets = self.tokenizer.encode_plus(
            target_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": targets["input_ids"].squeeze(0)
        }

In [4]:
class IterativeGenerativeRepair():
    def __init__(self, dirty_path, clean_path, exclude_list=None, iteration_number=3):
        self.dirty_df = pd.read_csv(dirty_path)
        self.clean_df = pd.read_csv(clean_path)
        self.clean_df.columns = self.dirty_df.columns
        self.device = 'cuda:0'
        self.iteration_number = iteration_number
        # exclude some unwanted columns like tuple_id
        if exclude_list:
            assert max(exclude_list) < len(self.clean_df.columns)
            self.clean_df.drop(self.clean_df.columns[exclude_list], axis=1, inplace=True)
            self.dirty_df.drop(self.dirty_df.columns[exclude_list], axis=1, inplace=True)
        
        self.error_df = (self.clean_df != self.dirty_df)
        self.res_df = self.clean_df.copy()  # store result sync
        self.temp_df = None  # copy res_df to process async

    
    def preprocess(self, target_index=0, iteration_time=1):
        train_data, test_data = [], []
        n_rows, n_cols = self.clean_df.shape
        column_names = list(self.clean_df.columns)
        
        target_column_name = column_names[target_index]
        input_column_name = column_names[:target_index]
        if target_index != n_cols-1:
            input_column_name += column_names[target_index+1:]
            
        if iteration_time!=1:
            # no need for masking then
            for row in range(n_rows):
                inputs = list(self.temp_df.iloc[row, :target_index].astype(str))
                # no index error
                if target_index != n_cols-1:
                    inputs += list(self.temp_df.iloc[row, target_index+1:].astype(str))
                input_content = ""
                for pair in zip(input_column_name, inputs):
                    input_content += pair[0] + ": " + pair[1] + "<extra_id_1>"
                post_fix = target_column_name + ": "
                input_content += post_fix

                # target_content = str(self.clean_df.iloc[row, -1]) 你无敌了
                target_content = str(self.clean_df.iloc[row, target_index])
                if self.error_df.iloc[row, target_index]:
                    test_data.append((input_content, target_content))
                else:
                    train_data.append((input_content, target_content))

        else:
            for row in range(n_rows):
                inputs = []
                for col in range(n_cols):
                    if col == target_index:
                        pass
                    else:
                        # ! column assigned incorrectly
                        if self.error_df.iloc[row, col]:  # if wrong then using mask
                            inputs.append("<extra_id_0>")
                        else:
                            inputs.append(str(self.dirty_df.iloc[row, col]))
                input_content = ""  # modify this initial value to implement prefix prompt
                # prefix = "some prompt"
                # input_content = prefix + input_content
                for pair in zip(input_column_name, inputs):
                    input_content += pair[0] + ": " + pair[1] + "<extra_id_1>"
                post_fix = target_column_name + ": "
                input_content += post_fix

                # split into test and train
                target_content = str(self.clean_df.iloc[row, target_index])
                if self.error_df.iloc[row, target_index]:
                    test_data.append((input_content, target_content))
                else:
                    train_data.append((input_content, target_content))
        
        return train_data, test_data


    # need modify
    def train_step(self, train_loader, epochs=3):
        model = self.model
        optimizer = self.optimizer
        device = self.device
        
        model.train()
        for epoch in range(epochs):
            print(f"Starting epoch {epoch+1}")
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                inputs = batch["input_ids"].to(device)
                labels = batch["labels"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                outputs = model(input_ids=inputs, labels=labels, attention_mask=attention_mask)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            print(f"Epoch {epoch+1} Train Loss: {total_loss/len(train_loader):.4f}")

    # need modify
    def test_step(self, test_loader, max_length=128):
        model = self.model
        device = self.device
        tokenizer = self.tokenizer
        
        model.eval()
        total_loss = 0
        predictions = []
        targets = []
        print('start testing')
        with torch.no_grad():
            for batch in test_loader:
                # loss part
                inputs = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                
                outputs = model(input_ids=inputs, labels=labels, attention_mask=attention_mask)
                loss = outputs.loss
                total_loss += loss.item()
                # prediction part
                generated_ids = model.generate(input_ids=inputs, attention_mask=attention_mask, max_length=max_length)
                # generated_ids = model.generate(input_ids=inputs, attention_mask=attention_mask, max_new_tokens=128)
                preds = [tokenizer.decode(g_id, skip_special_tokens=True, clean_up_tokenization_spaces=True) for g_id in generated_ids]
                predictions.extend(preds)
                # compare to targets
                targets.extend([tokenizer.decode(tgt, skip_special_tokens=True, clean_up_tokenization_spaces=True) for tgt in labels])
                
        print(f"Eval Loss: {total_loss/len(test_loader):.4f}")
        return predictions, targets

    def run(self, epochs=3, batch_size=16, model='base'):
        # if model == 'base':
        #     seq2seq_model = 'google-t5/t5-base'
        #     self.model = AutoModelForSeq2SeqLM.from_pretrained(seq2seq_model).to(device)
        # elif model == 'large':
        #     seq2seq_model = 'google-t5/t5-large'
        #     self.model = AutoModelForSeq2SeqLM.from_pretrained(seq2seq_model).to(device)
        # elif model == 'fast':
        #     seq2seq_model = 't5-small'  ## only for test run
        #     # seq2seq_model = 'google-t5/t5-large'
        #     self.model = AutoModelForSeq2SeqLM.from_pretrained(seq2seq_model)
        #     self.model = export_and_get_onnx_model(self.model).to(device)
        seq2seq_model = 'google-t5/t5-large'
        self.model = AutoModelForSeq2SeqLM.from_pretrained(seq2seq_model).to(device)
        self.epochs = epochs
        self.tokenizer = AutoTokenizer.from_pretrained(seq2seq_model)
        # self.model = AutoModelForSeq2SeqLM.from_pretrained(seq2seq_model).to(device)
        self.optimizer = AdamW(self.model.parameters(), lr=1e-4)
        self.batch_size = batch_size

        for iteration in range(self.iteration_number):
            # processing each column
            for ind in range(len(self.clean_df.columns)):
            # skip all true but useful cols
                if self.error_df.iloc[:, ind].sum() == 0:
                    continue
                train_data, test_data = self.preprocess(target_index=ind, iteration_time=iteration+1)
                train_dataset = Seq2SeqDataset(train_data, self.tokenizer)
                test_dataset = Seq2SeqDataset(test_data, self.tokenizer)
                train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=False)  # why need shuffle?
                test_loader = DataLoader(test_dataset, batch_size=self.batch_size, shuffle=False)
    
                print(f'---------------start fine tune col index {ind}---------------')
                self.train_step(train_loader, epochs=self.epochs)
                predictions, targets = self.test_step(test_loader)  # max_length 128
                print(f'---------------fine tune for col index {ind} done---------------')
                for i in range(5):
                    print(f"prediction: {predictions[i]}\t targets: {targets[i]}")
    
                # modify the data
                n_rows, n_cols = self.clean_df.shape
                pointer = 0
                for row in range(n_rows):
                    if self.error_df.iloc[row, ind]:
                        self.res_df.iloc[row, ind] = predictions[pointer]
                        pointer += 1
                self.temp_df = self.res_df.copy()

        torch.save(
            model.state_dict(),
            'test_saving.bin'
        )
        print('Done.')
        

    def get_res(self):
        return self.res_df
        

In [6]:
dirty_path = '../dataset/flight/dirty.csv'
clean_path = '../dataset/flight/clean.csv'
a = IterativeGenerativeRepair(dirty_path, clean_path, exclude_list=[0])
d = a.dirty_df
c = a.clean_df
e = a.error_df

In [7]:
print(d.head())
print(c.head())
print(e.head())

  src           flight sched_dep_time act_dep_time sched_arr_time act_arr_time
0  aa  AA-3859-IAH-ORD      7:10 a.m.    7:16 a.m.      9:40 a.m.    9:32 a.m.
1  aa  AA-1733-ORD-PHX      7:45 p.m.    7:58 p.m.     10:30 p.m.          NaN
2  aa  AA-1640-MIA-MCO      6:30 p.m.          NaN      7:25 p.m.          NaN
3  aa   AA-518-MIA-JFK      6:40 a.m.    6:54 a.m.      9:25 a.m.    9:28 a.m.
4  aa  AA-3756-ORD-SLC     12:15 p.m.   12:41 p.m.      2:45 p.m.    2:50 p.m.
  src           flight sched_dep_time act_dep_time sched_arr_time act_arr_time
0  aa  AA-3859-IAH-ORD      7:10 a.m.    7:16 a.m.      9:40 a.m.    9:32 a.m.
1  aa  AA-1733-ORD-PHX      7:45 p.m.    7:58 p.m.     10:30 p.m.   10:30 p.m.
2  aa  AA-1640-MIA-MCO      6:30 p.m.    6:30 p.m.      7:25 p.m.    7:25 p.m.
3  aa   AA-518-MIA-JFK      6:40 a.m.    6:54 a.m.      9:25 a.m.    9:28 a.m.
4  aa  AA-3756-ORD-SLC     12:15 p.m.   12:41 p.m.      2:45 p.m.    2:50 p.m.
   tuple_id    src  flight  sched_dep_time  act_dep_

In [9]:
# iteration number=1 epochs=3 approximately 6min
dirty_path = '../dataset/flight/dirty.csv'
clean_path = '../dataset/flight/clean.csv'
a = IterativeGenerativeRepair(dirty_path, clean_path, exclude_list=[0])
a.run()
r = a.get_res()

---------------start fine tune col index 2---------------
Starting epoch 1


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch 1 Train Loss: 1.0862
Starting epoch 2
Epoch 2 Train Loss: 0.0616
Starting epoch 3
Epoch 3 Train Loss: 0.0292
start testing
Eval Loss: 0.0358
---------------fine tune for col index 2 done---------------
prediction: 7:16 a.m.	 targets: 9:32 a.m.
prediction: 6:54 a.m.	 targets: 9:28 a.m.
prediction: 10:50 a.m.	 targets: 11:58 a.m.
prediction: 12:50 p.m.	 targets: 9:43 a.m.
prediction: 10:50 p.m.	 targets: 10:30 p.m.
---------------start fine tune col index 3---------------
Starting epoch 1
Epoch 1 Train Loss: 0.0176
Starting epoch 2
Epoch 2 Train Loss: 0.0161
Starting epoch 3
Epoch 3 Train Loss: 0.0148
start testing
Eval Loss: 0.0383
---------------fine tune for col index 3 done---------------
prediction: 7:25 p.m.	 targets: 7:25 p.m.
prediction: 11:25 p.m.	 targets: 6:55 a.m.
prediction: 11:22 p.m.	 targets: 5:50 a.m.
prediction: 9:23 p.m.	 targets: 12:15 a.m.
prediction: 11:50 p.m.	 targets: 11:50 p.m.
---------------start fine tune col index 4---------------
Starting epoch 1
Epoc

In [10]:
true = a.clean_df
new_error = (r != true)
print(r.head())
for i in range(6):
    print(a.error_df.iloc[:, i].sum(), new_error.iloc[:, i].sum())

  src           flight sched_dep_time act_dep_time sched_arr_time act_arr_time
0  aa  AA-3859-IAH-ORD      7:10 a.m.    7:16 a.m.      9:40 a.m.    9:32 a.m.
1  aa  AA-1733-ORD-PHX      7:45 p.m.    7:58 p.m.     10:30 p.m.   10:06 p.m.
2  aa  AA-1640-MIA-MCO      6:30 p.m.    7:25 p.m.      7:25 p.m.    7:15 p.m.
3  aa   AA-518-MIA-JFK      6:40 a.m.    6:54 a.m.      9:25 a.m.    9:28 a.m.
4  aa  AA-3756-ORD-SLC     12:15 p.m.   12:41 p.m.      2:45 p.m.    2:50 p.m.
0 0
0 0
911 907
1558 1529
1100 1068
1351 1270


In [37]:
# iteration=3, epochs=3, time 16min
dirty_path = '../dataset/flight/dirty.csv'
clean_path = '../dataset/flight/clean.csv'
a = IterativeGenerativeRepair(dirty_path, clean_path, exclude_list=[0])
a.run()

---------------start fine tune col index 2---------------
Starting epoch 1
Epoch 1 Train Loss: 0.9170
Starting epoch 2
Epoch 2 Train Loss: 0.0375
Starting epoch 3
Epoch 3 Train Loss: 0.0254
start testing
Eval Loss: 0.0356
---------------fine tune for col index 2 done---------------
prediction: 7:16 a.m.	 targets: 9:32 a.m.
prediction: 5:50 a.m.	 targets: 9:28 a.m.
prediction: 5:50 p.m.	 targets: 11:58 a.m.
prediction: 5:50 p.m.	 targets: 9:43 a.m.
prediction: 5:50 p.m.	 targets: 10:30 p.m.
---------------start fine tune col index 3---------------
Starting epoch 1
Epoch 1 Train Loss: 0.0164
Starting epoch 2
Epoch 2 Train Loss: 0.0145
Starting epoch 3
Epoch 3 Train Loss: 0.0129
start testing
Eval Loss: 0.0390
---------------fine tune for col index 3 done---------------
prediction: 7:25 p.m.	 targets: 7:25 p.m.
prediction: 11:33 p.m.	 targets: 6:55 a.m.
prediction: 11:33 p.m.	 targets: 5:50 a.m.
prediction: 9:33 p.m.	 targets: 12:15 a.m.
prediction: 11:50 p.m.	 targets: 11:50 p.m.
-------

In [39]:
r = a.get_res()
true = a.clean_df
new_error = (r != true)
print(r.head())
for i in range(6):
    print(a.error_df.iloc[:, i].sum(), new_error.iloc[:, i].sum())

  src           flight sched_dep_time act_dep_time sched_arr_time act_arr_time
0  aa  AA-3859-IAH-ORD      7:10 a.m.    7:16 a.m.      9:40 a.m.    9:32 a.m.
1  aa  AA-1733-ORD-PHX      7:45 p.m.    7:58 p.m.     10:30 p.m.   10:30 p.m.
2  aa  AA-1640-MIA-MCO      6:30 p.m.    7:25 p.m.      7:25 p.m.    7:25 p.m.
3  aa   AA-518-MIA-JFK      6:40 a.m.    6:54 a.m.      9:25 a.m.    9:28 a.m.
4  aa  AA-3756-ORD-SLC     12:15 p.m.   12:41 p.m.      2:45 p.m.    2:50 p.m.
0 0
0 0
911 911
1558 1557
1100 1078
1351 681


In [6]:
%%time
# iteration=3, epochs=3, using t5-large
dirty_path = '../dataset/flight/dirty.csv'
clean_path = '../dataset/flight/clean.csv'
a = IterativeGenerativeRepair(dirty_path, clean_path, exclude_list=[0])
# OOM, so batch size 4
a.run(batch_size=4)

---------------start fine tune col index 2---------------
Starting epoch 1


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch 1 Train Loss: 0.5867
Starting epoch 2
Epoch 2 Train Loss: 0.0217
Starting epoch 3
Epoch 3 Train Loss: 0.0187
start testing
Eval Loss: 0.0372
---------------fine tune for col index 2 done---------------
prediction: 	 targets: 9:32 a.m.
prediction: 	 targets: 9:28 a.m.
prediction: 	 targets: 11:58 a.m.
prediction: 	 targets: 9:43 a.m.
prediction: 	 targets: 10:30 p.m.
---------------start fine tune col index 3---------------
Starting epoch 1
Epoch 1 Train Loss: 0.0119
Starting epoch 2
Epoch 2 Train Loss: 0.0089
Starting epoch 3
Epoch 3 Train Loss: 0.0069
start testing
Eval Loss: 0.0418
---------------fine tune for col index 3 done---------------
prediction: 8:06 a.m.	 targets: 7:25 p.m.
prediction: 2:27 p.m.	 targets: 6:55 a.m.
prediction: 3:23 p.m.	 targets: 5:50 a.m.
prediction: 9:45 p.m.	 targets: 12:15 a.m.
prediction: 11:50 p.m.	 targets: 11:50 p.m.
---------------start fine tune col index 4---------------
Starting epoch 1
Epoch 1 Train Loss: 0.0166
Starting epoch 2
Epoch 2 Tr

In [7]:
r = a.get_res()
true = a.clean_df
new_error = (r != true)
print(r.head())
for i in range(6):
    print(a.error_df.iloc[:, i].sum(), new_error.iloc[:, i].sum())

  src           flight sched_dep_time act_dep_time sched_arr_time act_arr_time
0  aa  AA-3859-IAH-ORD      7:10 a.m.    7:16 a.m.      9:40 a.m.    9:32 a.m.
1  aa  AA-1733-ORD-PHX      7:45 p.m.    7:58 p.m.     10:30 p.m.   10:30 p.m.
2  aa  AA-1640-MIA-MCO      6:30 p.m.    7:25 p.m.      7:25 p.m.    7:25 p.m.
3  aa   AA-518-MIA-JFK      6:40 a.m.    6:54 a.m.      9:25 a.m.    9:28 a.m.
4  aa  AA-3756-ORD-SLC     12:15 p.m.   12:41 p.m.      2:45 p.m.    2:50 p.m.
0 0
0 0
911 911
1558 1558
1100 947
1351 169


In [5]:
%%time
# iteration=3, epochs=3, using t5-large, bug fixed
dirty_path = '../dataset/flight/dirty.csv'
clean_path = '../dataset/flight/clean.csv'
a = IterativeGenerativeRepair(dirty_path, clean_path, exclude_list=[0])
# OOM, so batch size 4
a.run(batch_size=4)

---------------start fine tune col index 2---------------
Starting epoch 1


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch 1 Train Loss: 0.5859
Starting epoch 2
Epoch 2 Train Loss: 0.0331
Starting epoch 3
Epoch 3 Train Loss: 0.0258
start testing
Eval Loss: 0.0373
---------------fine tune for col index 2 done---------------
prediction: 7:00 a.m.	 targets: 7:10 a.m.
prediction: 6:40 a.m.	 targets: 6:40 a.m.
prediction: 	 targets: 8:00 a.m.
prediction: 	 targets: 8:41 a.m.
prediction: 	 targets: 7:45 p.m.
---------------start fine tune col index 3---------------
Starting epoch 1
Epoch 1 Train Loss: 0.0418
Starting epoch 2
Epoch 2 Train Loss: 0.0266
Starting epoch 3
Epoch 3 Train Loss: 0.0145
start testing
Eval Loss: 0.0216
---------------fine tune for col index 3 done---------------
prediction: 6:57 p.m.	 targets: 6:30 p.m.
prediction: 11:56 p.m.	 targets: 11:25 p.m.
prediction: 11:55 a.m.	 targets: 11:55 p.m.
prediction: 9:23 p.m.	 targets: 9:00 p.m.
prediction: 8:23 p.m.	 targets: 8:25 p.m.
---------------start fine tune col index 4---------------
Starting epoch 1
Epoch 1 Train Loss: 0.0324
Starting e

In [6]:
r = a.get_res()
true = a.clean_df
new_error = (r != true)
print(r.head())
for i in range(6):
    print(a.error_df.iloc[:, i].sum(), new_error.iloc[:, i].sum())

  src           flight sched_dep_time act_dep_time sched_arr_time act_arr_time
0  aa  AA-3859-IAH-ORD      7:10 a.m.    7:16 a.m.      9:40 a.m.    9:32 a.m.
1  aa  AA-1733-ORD-PHX      7:45 p.m.    7:58 p.m.     10:30 p.m.   10:30 p.m.
2  aa  AA-1640-MIA-MCO      6:30 p.m.    6:57 p.m.      7:25 p.m.    7:25 p.m.
3  aa   AA-518-MIA-JFK      6:40 a.m.    6:54 a.m.      9:25 a.m.    9:28 a.m.
4  aa  AA-3756-ORD-SLC     12:15 p.m.   12:41 p.m.      2:45 p.m.    2:50 p.m.
0 0
0 0
911 500
1558 570
1100 405
1351 436


In [8]:
r.to_csv('result_t5-large.csv', index=False)

In [6]:
dirty_path = '../dataset/flight/dirty.csv'
clean_path = '../dataset/flight/clean.csv'
a = IterativeGenerativeRepair(dirty_path, clean_path, exclude_list=[0])
# OOM, so batch size 4
a.run(batch_size=16, model='fast')

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

OSError: Incorrect path_or_model_id: 'T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bia

In [2]:
from datetime import datetime as dt
dt.now()

datetime.datetime(2025, 3, 10, 13, 9, 26, 208556)

In [5]:
# demo for fastT5
from fastT5 import export_and_get_onnx_model
from transformers import AutoTokenizer

model_name = 't5-small'
model = export_and_get_onnx_model(model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name)
t_input = "translate English to French: The universe is a dark forest."
token = tokenizer(t_input, return_tensors='pt')

tokens = model.generate(input_ids=token['input_ids'],
               attention_mask=token['attention_mask'],
               num_beams=2)

output = tokenizer.decode(tokens.squeeze(), skip_special_tokens=True)
print(output)

Exporting to onnx... |################################| 3/3


TypeError: quantize_dynamic() got an unexpected keyword argument 'activation_type'

In [5]:
%%time
# iteration=3, epochs=3, using t5-large, bug fixed
dirty_path = '../dataset/flight/dirty.csv'
clean_path = '../dataset/flight/clean.csv'
a = IterativeGenerativeRepair(dirty_path, clean_path, exclude_list=[0])
# OOM, so batch size 4
a.run(batch_size=4)

---------------start fine tune col index 2---------------
Starting epoch 1


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch 1 Train Loss: 0.5793
Starting epoch 2


KeyboardInterrupt: 

In [3]:
torch.cuda.empty_cache()

NameError: name 'torch' is not defined

In [6]:
import pandas as pd
dirty_path = '../dataset/hospital/dirty.csv'
clean_path = '../dataset/hospital/clean.csv'
clean = pd.read_csv(clean_path)
dirty = pd.read_csv(dirty_path)
clean.columns = dirty.columns
clean = clean.astype(str)
print(clean.dtypes, dirty.dtypes)

index                object
provider_number      object
name                 object
address_1            object
address_2            object
address_3            object
city                 object
state                object
zip                  object
county               object
phone                object
type                 object
owner                object
emergency_service    object
condition            object
measure_code         object
measure_name         object
score                object
sample               object
state_average        object
dtype: object index                 int64
provider_number      object
name                 object
address_1            object
address_2            object
address_3            object
city                 object
state                object
zip                  object
county               object
phone                object
type                 object
owner                object
emergency_service    object
condition            object
measur